# Magic commands
- %sh - bash commands
- %python, %scala, %r, %sql - using specific programming language for cell
- %fs - same as dbutils using filesystem commands
- %md - markdown language 
- %run - run external notebook file
- %pip - install library uing pip 


# Using display and displayHTML

- display(df) - make an table and show dataframe
- displayHTML(html_code) - represent an HTML formatted code


# Markdown syntax
- [Click to redirect Markdown Guide](https://www.markdownguide.org/extended-syntax/)

# Widgets



In [0]:
dbutils.widgets.text('TextWidgetName', 'DefaultValue')
dbutils.widgets.dropdown('DropdownWidgetName', '1', [str(i) for i in range(1, 10)])
dbutils.widgets.combobox('ComboBoxWidgetName', 'A', ['A', 'B', 'C'])
dbutils.widgets.multiselect('MultiSelectWidgetName', 'Yes', ['Yes', 'No', 'Maybe'])
dbutils.widgest.removeAll()

In [ ]:
%%sql
%sql
CREATE WIDGET TEXT state DEFAULT 'CA'; -- Creating a widget type Text with default value 'DefaultValue'

In [ ]:
%%sql
%sql
REMOVE WIDGET state; -- Removing the widget

# Using params in SQL condition



In [ ]:
%%sql
%sql
SELECT * FROM events WHERE geo.state = getArgument('state')


# Access to filesystem DBFS

- %fs ls - show files
- %fs ls /direcory - show all files in directory
- %fs head /dir/file_name.ext - show head of the file
- %fs mounts - show mounted drives
- %fs help - show help for dbutils


# Spark Core

In [ ]:
%%sql
%sql
SELECT id, result FROM exams WHERE result > 70 ORDER BY result;

In [0]:
%python
df = spark.table('exams').select('id', 'result').where('result > 70').orderBy('result')
display(df)

In [0]:
# DataFrame reader

parquet_df = spark.read.parquet('/mnt/training/user/users.parquet')

csv_df = spark.read.option('header', 'true').option('sep', ',').option('inferSchema', 'true').csv('/mnt/training/user/users.csv')
csv_df = spark.read.options(header='true', sep=',', inferSchema='true').csv('/mnt/training/user/users.csv')
csv_df = spark.read.csv('/mnt/training/user/users.csv', header='true', sep=',', inferSchema='true')

In [0]:
# Excplicit VS Implicit VS Infer schema

# Explicit schema - define schema by string - without Reading file
ddl_schema = 'country STRING, age INT, name STRING, salary DOUBLE'
user_df = spark.read.schema(ddl_schema).csv('/mnt/training/user/users.csv')

# Implicit schema - create default coulmn name - without Reading file
user_df = spark.read.load('/mnt/training/user/users.csv', format='csv', header=False)

# Infer schema - infer schema from data by Reading file
user_df = spark.read.load('/mnt/training/user/users.csv', format='csv', inferSchema=True, header=True)

# DataFrame Writer

df.write.option('compression', 'snappy').mode('overwrite).parquet('/dir/filepath')

df.write.mode('append').saveAsTable('table_name')

In [0]:
%python

result_df = spark.sql("SELECT * FROM exams WHERE result > 70 ORDER BY result")
display(result_df)
result_df.printSchema() # display schema
result_df.count() # count rows
result_df.summary() # summary statistics
result_df.head(5) # display first 5 rows
result_df.show(5) # display first 5 rows
result_df.createOrReplaceTempView('exams_result') # create temp view

# Manual Schema 

In [0]:
%python

from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType

ddl = StructType([
    StructField('user_id', IntegerType(), True),
    StructField('user_name', StringType(), True),
    StructField('user_age', IntegerType(), True),
    StructField('user_salary', DoubleType(), True)
    ])

user_df = spark.read.schema(ddl).csv('/mnt/training/user/users.csv')

In [0]:
# read JSON
events_df = spark.read.option('inferSchema', 'true').json('/mnt/training/events/events.json')

In [0]:
%python
# Save to Delta format
events_df.write.format('delta').mode('overwrite').save('/mnt/training/events/events_delta')

# Column Expressions

In [0]:
%python

from pyspark.sql.functions import col

print(event_df['column_name'])
print(event_df.column_name)
print(col('column_name'))

col('column_name').alias('new_column_name')

user_df = user_df.withColumn('salaryPerYear', col('salary') * 12) # add new column with new name by alias
user_df = user_df.withColumn('salaryAndBonus', col('salary') * col('bonus')) # add new column with new name by alias


user_df = (user_df.filter(col('user_id').isNotNull())
                .filter(col('salary') > 10000)
                .withColumn('salaryPerYear', col('salary') * 12)
                .withColumn('salaryAndBonus', col('salary') * col('bonus'))
                .sort(col('salary').desc()))

# using selectExpr
user_df = user_df.selectExpr('*', 
                            'salary * 12 as salaryPerYear',
                            'salary * bonus as salaryAndBonus',
                            "device in ('Mac', 'IOS') as apple_user")

mobile_df = user_df.withColumn('apple_user', col('device').isin(['Mac', 'IOS']))

# Column renaming
user_df = user_df.withColumnRenamed('user_id', 'id')

# Filtering dataframe
user_df = user_df.where((col('user_id') > 10) & (col('user_age') > 20)&(col('user_salary') > 10000)&(col('email').isNotNull()))

# Drop Duplicates
user_df = user_df.dropDuplicates() # drop duplicates
user_df = user_df.dropDuplicates(['user_id']) # drop duplicates by column
user_df = user_df.disinct() # drop duplicates

# using limit
user_df = user_df.limit(10) # limit rows by 10 records

# sorting dataframe
user_df = user_df.sort(col('user_id').desc()) # sort by column, same as orderBy(col('user_id').desc())
user_df = user_df.orderBy(col('user_id').desc(), col('user_age'))

# Functions in Spark

In [0]:
# Aggregation
from pyspark.sql import functions as F

bonus_df = (top_traffic_df.groupBy('device')
                        .agg(F.sum('bonus').alias('total_bonus'))
                        .withColumn('total_bonus_2dec', F.format_number('total_bonus', 2)))

city_purchase_by_quantities_df = df.groupBy('city', 'purchase_quantity').sum('purchase_amount', 'purchase_quantity')

# Possible functions for groupBy is sum, avg, min, max, count, countDistinct, first, last, pivot
# Possible functions for aggregations is: sum, avg, min, max, stddev, variance, approx_count_distinct

state_purchase_df = (df.groupBy('state')
                      .agg(
                            F.sum('purchase_amount').alias('total_purchase'),
                            F.avg('purchase_quantity').alias('avg_quantity'),
                            F.min('purchase_quantity').alias('min_quantity'),
                            F.max('purchase_quantity').alias('max_quantity')
                            ))
                      
# Datetime 
# https://docs.databricks.com/en/sql/language-manual/sql-ref-datetime-pattern.html

# Tip: convert unixtime to timestamp
df2 = df1.withColumn('ts', (F.col('unixtime')/1e6).cast('timestamp'))

# Datetime functions -> add_months, current_date, date_add, date_sub, datediff, months_between, to_date, to_timestamp

formatted_df = (timestap_df
                        .withColumn('date_string', F.date_format('ts', 'MMMM dd, yyyy')) # format date July 07, 2020
                        .withColumn('ts', F.date_format('ts', 'HH:mm:ss.SSSSSS')) #        format time 12:00:00.000000
                        .withColumn('date', F.to_date(col('ts'))) #                        format date 2020-07-07
                        .withColumn('plus_2_days', F.date_add(col('ts'), 2)) #             format date 2020-07-09
                )
# Complex Types
# Built functions for string -> translate, regex_replace, regex_extract, ltrim, lower, upper, split
email_df = df.select(F.split(F.col('email'), '@').getItem(0).alias('email_domain')) # split email and get first part john.doe@mail.com -> john.doe
# Built-in collection functions -> array_contains, explode, element_at, collect_set, slice
df.agg(F.collect_set('age')) # [2,4]
df.agg(F.collect_list('age')) # [2,2,4]

# Additional functions
# col, lit, isNull, isNotNull, rand
# join -> emp_df.join(supplier_df, emp_df.user_id == supplier_df.user_id, 'left')
# df.na.drop(), df.na.fill('value')

# UserDefinedFunctions
# Better use transform instead UDF
from pyspark.sql.functions import udf

def add_bonus(salary):
      return salary * 1.1

# example of UDF
add_bonus_udf = udf(add_bonus)
salary_df = user_df.withColumn('salary_bonus', add_bonus_udf(col('salary'))) # performance ~ 2 sec

# example of transform
salary_df = user_df.withColumn('salary_bonus', user_df.transform(lambda row: row.salary * 1.1)) # performance ~ 1 sec

In [ ]:
%%sql
-- Creating function using SQL
CREATE FUNCTION add_bonus(salary DOUBLE) RETURNS DOUBLE RETURN salary * 1.1;
SELECT add_bonus(10000) AS salary_bonus FROM dual;

# Architecture and Performance

In [0]:
# Architecture

# Narrow transformations -> select, union, filter, cast
# Wide transformations -> distinct, groupBy, sort, join

# CBO - cost based optimizer
spark.conf.set("spark.sql.cbo.enabled", "true")
spark.conf.set("spark.sql.cbo.joinReordering.enabled", "true")
spark.conf.set("spark.sql.statistics.histogram.enabled", "true")

# Performance and Query optimization
# Enable adaptive query optimization AQO
spark.conf.set('spark.sql.adaptive.enabled', 'true')
saprk.conf.set('spark.sql.adaptive.coalescePartitions.enabled', 'true')

# Caching
df.cache() # cache dataframe
df.count() # perform an action to trigger cache
df.unpersist() # remove cache

# Memory Partitioning
# Set number of partitions
spark.conf.set("spark.sql.shuffle.partitions", "1000")
total_num_partitions = df.rdd.getNumPartitions()
spark.conf.set("spark.sql.shuffle.partitions", str(total_num_partitions))

# Coase partitions
df.coalese(10)  # coalesce partitions down to 10, only for decrease
df.repartition(100)  # repartition dataframe up to 100 partitions, can decrease and increase partitions

# Structure Streaming

In [0]:
# Streaming Query

# Build a streaming dataframe
df = spark.readStream.schema(schema).option('maxFilesPerTrigger', '1').format('delta').load(eventPath)
df.isStreaming() # True

# Write streaming results
checkpoint_path = '/tmp/checkpoint'
output_path = '/tmp/output'

df = spark.readStream.schema(schema).option('maxFilesPerTrigger', '1').format('delta').load(eventPath)

user_email_df = df.withColumn('apple_user', col('device').isin(['Mac', 'IOS'])).withColumn('date', F.to_date(col('event_timestamp'))).withColumn('email', F.split(F.col('email'), '@').getItem(0)).select('user_id', 'email', 'date', 'apple_user')

device_query = user_email_df.writeStream.format('delta').queryName('user_email').trigger(processingTime='10 seconds').option('checkpointLocation', checkpoint_path).outputMode('append').start(output_path)

# Monitoring Streaming
device_query.id # id of streaming query like 1e2-161101_000000_000000
device_query.isActive # True/False
device_query.lastProgress # last progress of streaming query 
device_query.status # {'message': 'Waiting for more data.', 'isDataAvailable': False, 'isTriggerAvailable': False}
device_query.stop() # stop streaming query
device_query.awaitTermination() # wait for streaming query to stop

# Batch processing -> High throughput, Slow updates, Manual Bookkeeping

# Stream processing -> Low latency, Efficient updates, Automatic Bookkeeping on new data

# Source for streaming -> Kafka, Kinesis, File, Socket, HTTP, JDBC, REST, Flume, ZeroMQ
df = spark.readStream.option('maxFilesPerTrigger', '100').format('delta').load(eventPath)
df.createOrReplaceTempView('events')
spark.readStream.format('delta').table('events')
# Output modes -> Append (add new records), Complete (overwrite all records), Update (update records)
# Trigger types: Continuous, Once, ProcessingTime, EventTime
Fixed interval df.trigger(interval=10, processingTime='10 seconds')
Once df.trigger(once=True)
Available df.trigger(availableNow=True)
Continuous df.trigger(processingTime='10 seconds')

# Complete streaming query example
df = spark.readStream.schema(schema).option('maxFilesPerTrigger', '1').format('parquet').load(eventPath)
email_traffic_df = df.filter(F.col('traffic_source')== 'email').withColumn('mobile', F.col('device').IsIn(['IOS', 'Android'])).select('user_id', 'mobile', 'event_timestamp')
device_query_df = email_traffic_df.writeStream.outputMode('append').format('delta').queryName('email_traffic').trigger(processingTime='10 seconds').option('checkpointLocation', checkpointPath).start(outputPath)

# Streaming Aggregation
# Time-based windows

# Tumbling windows
stream_df.groupBy(F.window('time_column', "5 minutes")).count() # Adjacent 5 minutes window, any window events are aggregated into one 
stream_df.groupBy(
    F.col('device'), F.window(F.col('time_column'), "1 hour")
).count() # Group by device and 1 hour window

df = spark.createDataFrame([(1, '2020-01-01 10:00:00')]).toDF('id', 'time_column')
window_df = df.groupBy(F.window('time_column', "10 minutes")).agg(
    F.sum('id').alias('sum')
).select('sum', F.col('window.start').cast('string').alias('start'), F.col('window.end').cast('string').alias('end')) # Show start and end of each window


# Rolling or Sliding windows
stream_df.groupBy(F.window('time_column', "5 minutes", "10 minutes")).count() # Window overlap, 5 minutes window, 10 minutes sliding window

# Watermark - allows for update lated arriving data
# Count traffic_user in 1 Hour interval and 2 hours late data
df = df.withColumn("created_at", (F.col('event_timestamp')/1e6).cast('timestamp'))
       .withWatermark('created_at', '2 hours')
       .groupBy("traffic_source", F.window("created_at", "1 hour"))
       .agg(F.count("user_id").alias("active_users"))
# Real time Aggregations


# Delta Lake

In [0]:
%sql

DESCRIBE TABLE table_changes; -- Displaying the table changes

DESCRIBE HISTORY table_changes; -- Displaying the history of the table

SELECT * FROM table_changes@v1; -- Displaying the table changes at version 1

SELECT * FROM table_changes VERSION AS OF 1; -- Displaying the table changes at version 1

SELECT * FROM table_changes TIMESTAMP AS OF '2022-01-01 00:00:00'; -- Displaying the table changes at a specific timestamp

VACUUM table_changes RETAIN 1 HOURS; -- Vacuuming the table changes to retain the last hour of data

In [0]:
df = spark.read.format('delta')
                .option('timestampAsOf', '2020-01-01')
                .load('/mnt/training/flight-data/delta/2015-summary.delta') # Load data from 2015-summary.delta